In [10]:
import pandas as pd
import re
import sys
from datetime import datetime

def classify_priority(memo):
    """
    Classifies lead priority based on memo content.
    Returns: Priority Tier (Tier 1, Tier 2, Tier 3, or Low Priority)
    """
    if pd.isna(memo):
        return "Low Priority"

    text = str(memo).lower().strip()

    # 🥇 TIER 1: High-Intent Leads (proposal + commitment)
    tier1_patterns = [
        r'(proposal|invoice|quotation)\s+(sent|shared|provided).*?(will\s+let\s+me\s+know|will\s+get\s+back|will\s+call|will\s+review|will\s+inform|will\s+update|payment\s+(done|secured|will\s+make))',
        r'(will\s+let\s+me\s+know|will\s+get\s+back|will\s+call|will\s+review|will\s+inform|will\s+update|payment\s+(done|secured)).*?(proposal|invoice|quotation)',
        r'(payment|pay|invoice).*?(secured|done|will\s+make|will\s+do|tomorrow|today)',
        r'(interested|convinced|positive).*?(proposal|package).*?(will\s+let\s+know|will\s+get\s+back)',
        r'(discussed|spoke|connected).*?(packages|pricing|details).*?(will\s+let\s+know|will\s+get\s+back)',
        r'(payment\s+done|sold|secured\s+payment|invoice\s+sent)',
        r'(will\s+pay|payment\s+will\s+make|will\s+try\s+to\s+make\s+a\s+payment)'
    ]

    for pattern in tier1_patterns:
        if re.search(pattern, text):
            return "Tier 1"

    # 🥈 TIER 2: Active Hiring + Details Shared
    tier2_patterns = [
        r'(hiring|recruitment|vacancy|position|job).*?(active|ongoing|multiple|need|require|looking\s+for|has|have)',
        r'(have|has)\s+\d+\s+(hiring|vacancy|position|job)',
        r'(sent|shared|provided|gave|forwarded).*?(details|contact|number|email|hr|pic|jd)',
        r'(contact\s+details|hr\s+number|pic\s+number|email).*?(shared|provided|sent)',
        r'(need|require|looking\s+for)\s+\d+\s+(position|hiring|vacancy)',
        r'(multiple\s+hiring|multiple\s+positions|several\s+hiring)',
        r'(credentials|username|password)\s+(sent|shared)',
        r'(onboarded|created\s+account|free\s+onboard)',
        r'(forwarded\s+to\s+(hr|management|pic))'
    ]

    for pattern in tier2_patterns:
        if re.search(pattern, text):
            return "Tier 2"

    # 🥉 TIER 3: Busy but Committed to Callback
    tier3_patterns = [
        r'(busy|in\s+meeting|in\s+a\s+meeting|outside|driving|sleeping|in\s+leave|at\s+lunch|at\s+dinner).*?(call\s+later|will\s+call|asked\s+to\s+call|callback|recall|re-call)',
        r'(will\s+call\s+back|call\s+me\s+back|call\s+later|callback|recall).*?(busy|meeting)',
        r'(call\s+after|call\s+at|call\s+on|call\s+tomorrow|call\s+next|call\s+later|call\s+sunday|call\s+monday|call\s+tuesday|call\s+wednesday|call\s+thursday|call\s+friday|call\s+saturday)\s+(.*?\d+[:\d]*\s*(am|pm|today|tomorrow|next\s+week|next\s+month))?',
        r'(asked\s+to\s+call|requested\s+to\s+call|need\s+to\s+call).*?(later|tomorrow|next|after|sunday|monday|tuesday|wednesday|thursday|friday|saturday)',
        r'(in\s+meeting|busy).*?(will\s+talk\s+later|will\s+call\s+back|will\s+get\s+back)',
        r'(call\s+waiting|call\s+me\s+back|will\s+call)',
        r'(pic\s+was\s+busy|pic\s+in\s+a\s+meeting|pic\s+is\s+outside)'
    ]

    for pattern in tier3_patterns:
        if re.search(pattern, text):
            return "Tier 3"

    return "Low Priority"

def classify_lead(memo):
    """
    Classifies lead based on memo content using contextual keyword analysis.
    Returns: (status: str, reason: str)
    """
    if pd.isna(memo) or not str(memo).strip():
        return "Not Assignable", "Empty memo"

    text = str(memo).lower().strip()

    # 🔴 STRONG DISQUALIFIERS (immediate rejection)
    strong_disqualifiers = [
        r'wrong\s+number', r'switched\s+off', r'invalid\s+number', r'number\s+issue',
        r'did\s+not\s+pick', r'call\s+not\s+received', r'not\s+reachable', r'helpline',
        r'receptionist', r'customer\s+care', r'not\s+the\s+pic', r'job\s+seeker',
        r'candidate\s+looking', r'not\s+a\s+company', r'blocked\s+my\s+number',
        r'hung\s+up', r'rude', r'do\s+not\s+call', r'lost\s+lead', r'left\s+the\s+job',
        r'cut\s+the\s+call', r'not\s+interested', r'rejected', r'no\s+hiring\s+plan',
        r'hiring\s+done', r'no\s+need', r'no\s+hiring', r'business\s+closed',
        r'internally\s+hired', r'using\s+bdjobs', r'not\s+require', r'no\s+plan',
        r'no\s+response', r'call\s+dropped', r'network\s+issue', r'voice\s+not\s+audible',
        r'number\s+busy', r'call\s+waiting', r'call\s+forwarded', r'not\s+the\s+person',
        r'pic\s+not\s+available', r'pic\s+not\s+connected', r'pic\s+absent'
    ]
    for pattern in strong_disqualifiers:
        if re.search(pattern, text):
            match = re.search(pattern, text).group()
            return "Not Assignable", f"Invalid contact/rejection: '{match}'"

    # 🟢 POSITIVE CONVERSATION INDICATORS (requires evidence of 2-way engagement)
    positive_patterns = [
        r'(discussed|spoke|connected|conversation|responded|agreed|confirmed|scheduled)\s+(with|to|on)',
        r'(he|she|they)\s+(said|mentioned|asked|requested|will|let\s+me\s+know|provided|shared|forwarded|informed|talked)',
        r'(sent|shared|pitched|explained|provided)\s+(proposal|details|package|invoice|credentials|jd)',
        r'(busy|in\s+meeting|outside|in\s+a\s+meeting)\s+.*?(asked\s+to\s+call|will\s+call|call\s+later|callback)',
        r'permission\s+(granted|done)', r'will\s+(follow\s+up|review|discuss|get\s+back|inform|update|let\s+know|call\s+back)',
        r'(interested|positive|convinced|will\s+pay|payment\s+done|secured\s+payment)', r'meeting\s+(done|scheduled|set)',
        r'(provided|gave|shared|forwarded)\s+(number|email|contact|hr|pic|details)', r'forwarded\s+to\s+(hr|management)',
        r'(credentials|username|password)\s+(sent|shared|resent)', r'(onboarded|created\s+account)',
        r'(need|require|looking\s+for)\s+(hiring|position|vacancy)', r'(has|have)\s+hiring',
        r'(call\s+later|callback|will\s+call\s+back)', r'(sent\s+proposal|proposal\s+sent|shared\s+proposal)',
        r'(interested|receptive|positive\s+response)', r'(will\s+talk|will\s+discuss|will\s+review)',
        r'(payment\s+done|sold|secured\s+payment|invoice\s+sent)',
        r'(forwarded\s+proposal|shared\s+details|provided\s+information)',
        r'(will\s+let\s+know|will\s+get\s+back|will\s+inform|will\s+update)'
    ]
    for pattern in positive_patterns:
        if re.search(pattern, text):
            # ⚠️ Check for overriding rejection AFTER positive signal
            rejection_override = [
                r'not\s+interested\s+(now|anymore)', r'hiring\s+done', r'no\s+need\s+(now|right\s+now)',
                r'internally\s+hired', r'rejected\s+by\s+pic', r'budget\s+issue.*not\s+proceeding',
                r'no\s+hiring\s+plan', r'no\s+plan\s+to\s+hire', r'not\s+require\s+right\s+now',
                r'hired\s+internally', r'completed\s+hiring'
            ]
            if any(re.search(r, text) for r in rejection_override):
                return "Not Assignable", "Positive interaction overridden by rejection"
            return "Assignable", f"Valid conversation: '{re.search(pattern, text).group()}'"

    return "Not Assignable", "No evidence of actual conversation"

def extract_followup_timing(memo):
    """
    Extracts follow-up timing information from memo.
    Returns: Timing string or "Not specified"
    """
    if pd.isna(memo):
        return "Not specified"

    text = str(memo).lower()

    # Look for timing patterns
    timing_patterns = [
        r'call\s+(after|at)\s+(\d+[:\d]*\s*(am|pm))',
        r'call\s+(tomorrow|today|next\s+week|next\s+month|after\s+eid|after\s+lunch|after\s+dinner|sunday|monday|tuesday|wednesday|thursday|friday|saturday)',
        r'(tomorrow|today|next\s+week|next\s+month|after\s+eid|after\s+lunch|after\s+dinner|sunday|monday|tuesday|wednesday|thursday|friday|saturday)\s+(need|must)\s+follow',
        r'call\s+on\s+(\d+[-/]\d+[-/]\d+)',
        r'call\s+at\s+(\d+[:\d]*\s*(am|pm))',
        r'after\s+(\d+)\s+(min|minute|hour|pm|am)',
        r'(sunday|monday|tuesday|wednesday|thursday|friday|saturday)',
        r'\d{1,2}[:\d]+\s*(am|pm)',
        r'(next|this)\s+(week|month)',
        r'call\s+back\s+(later|tomorrow|next\s+week)',
        r'(need|must)\s+follow\s+(up|tomorrow|next\s+week)'
    ]

    for pattern in timing_patterns:
        match = re.search(pattern, text)
        if match:
            return match.group(0).strip()

    return "Not specified"

def main():
    try:
          # Load data with encoding fallbacks
        try:
            df = pd.read_csv('C:/Users/Asus/All Lead Memo Only Data.csv', encoding='utf-8')
        except UnicodeDecodeError:
            # Added the missing comma here 👇
            df = pd.read_csv('C:/Users/Asus/All Lead Memo Only Data.csv', encoding='latin1')

        # Validate columns
        required_cols = ['Lead No.', 'memo']
        if not all(col in df.columns for col in required_cols):
            # Try common alternative column names
            alt_cols = {
                'Lead No.': ['lead no', 'lead_no', 'leadno', 'id'],
                'memo': ['remarks', 'notes', 'comment', 'activity']
            }
            found_cols = {}
            for std, alts in alt_cols.items():
                for col in df.columns:
                    if col.lower().replace(' ', '_').replace('.', '') in [std.lower().replace(' ', '_').replace('.', '')] + [a.lower() for a in alts]:
                        found_cols[std] = col
                        break
            if len(found_cols) < 2:
                print("❌ ERROR: Required columns not found. Expected 'Lead No.' and 'memo'.")
                print(f"   Available columns: {list(df.columns)}")
                sys.exit(1)
            df = df.rename(columns={found_cols['Lead No.']: 'Lead No.', found_cols['memo']: 'memo'})

        print("="*80)
        print("🚀 TELE-SALES LEAD PRIORITIZATION SYSTEM")
        print("="*80)

        # Classify leads
        df[['Assignment Status', 'Reason']] = df['memo'].apply(
            lambda x: pd.Series(classify_lead(x))
        )

        # Add priority tier for assignable leads
        df['Priority Tier'] = df.apply(
            lambda row: classify_priority(row['memo']) if row['Assignment Status'] == 'Assignable' else 'N/A',
            axis=1
        )

        # Add follow-up timing
        df['Follow-up Timing'] = df.apply(
            lambda row: extract_followup_timing(row['memo']) if row['Assignment Status'] == 'Assignable' else 'N/A',
            axis=1
        )

        # Filter and sort assignable leads
        assignable = df[df['Assignment Status'] == 'Assignable'].sort_values('Lead No.')

        # Generate report
        print(f"\n📊 Total leads analyzed: {len(df):,}")
        print(f"✅ Assignable to tele-sales: {len(assignable):,} ({len(assignable)/len(df)*100:.1f}%)")
        print(f"❌ Not assignable: {len(df) - len(assignable):,}")

        if len(assignable) > 0:
            # Priority distribution
            priority_counts = assignable['Priority Tier'].value_counts()
            print("\n🎯 PRIORITY TIER DISTRIBUTION:")
            print("-" * 80)
            for tier in ['Tier 1', 'Tier 2', 'Tier 3', 'Low Priority']:
                count = priority_counts.get(tier, 0)
                percentage = (count / len(assignable) * 100) if len(assignable) > 0 else 0
                tier_emoji = {'Tier 1': '🥇', 'Tier 2': '🥈', 'Tier 3': '🥉', 'Low Priority': '📝'}[tier]
                print(f"{tier_emoji} {tier:<15} : {count:>6,} leads ({percentage:>5.1f}%)")

            # Detailed breakdown by priority
            print("\n📋 DETAILED LEAD BREAKDOWN:")
            print("="*80)

            for tier in ['Tier 1', 'Tier 2', 'Tier 3']:
                tier_leads = assignable[assignable['Priority Tier'] == tier].head(5)
                if len(tier_leads) > 0:
                    total_count = len(assignable[assignable['Priority Tier'] == tier])
                    print(f"\n{tier} LEADS ({total_count:,} total):")
                    print("-" * 80)
                    for _, row in tier_leads.iterrows():
                        memo_preview = str(row['memo'])[:80] + "..." if len(str(row['memo'])) > 83 else str(row['memo'])
                        timing = row['Follow-up Timing'] if row['Follow-up Timing'] != "Not specified" else ""
                        print(f"Lead No.: {row['Lead No.']:<8} | Priority: {row['Priority Tier']:<10} | Timing: {timing:<20}")
                        print(f"          Memo: {memo_preview}")
                        print("-" * 80)

            # Generate timestamp for unique filename
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

            # Export CSV with priority information
            output_file = f'assignable_leads_{timestamp}.csv'
            try:
                assignable[['Lead No.', 'Priority Tier', 'Follow-up Timing', 'Reason']].to_csv(output_file, index=False)

                # Export priority-specific files
                tier1_file = f'priority_tier_1_leads_{timestamp}.csv'
                tier1_leads = assignable[assignable['Priority Tier'] == 'Tier 1'][['Lead No.', 'Priority Tier', 'Follow-up Timing', 'Reason', 'memo']]
                tier1_leads.to_csv(tier1_file, index=False)

                tier2_file = f'priority_tier_2_leads_{timestamp}.csv'
                tier2_leads = assignable[assignable['Priority Tier'] == 'Tier 2'][['Lead No.', 'Priority Tier', 'Follow-up Timing', 'Reason', 'memo']]
                tier2_leads.to_csv(tier2_file, index=False)

                tier3_file = f'priority_tier_3_leads_{timestamp}.csv'
                tier3_leads = assignable[assignable['Priority Tier'] == 'Tier 3'][['Lead No.', 'Priority Tier', 'Follow-up Timing', 'Reason', 'memo']]
                tier3_leads.to_csv(tier3_file, index=False)

                print(f"\n💾 EXPORT SUCCESSFUL:")
                print(f"   • Main file: '{output_file}' ({len(assignable):,} leads)")
                print(f"   • Tier 1 file: '{tier1_file}' ({len(tier1_leads):,} leads)")
                print(f"   • Tier 2 file: '{tier2_file}' ({len(tier2_leads):,} leads)")
                print(f"   • Tier 3 file: '{tier3_file}' ({len(tier3_leads):,} leads)")

                # Also print CSV content to console for easy copy-paste
                print("\n📋 CSV CONTENT (copy-paste ready):")
                print("="*80)
                csv_content = assignable[['Lead No.', 'Priority Tier', 'Follow-up Timing']].to_csv(index=False)
                print(csv_content[:2000])  # Print first 2000 characters
                if len(csv_content) > 2000:
                    print(f"... ({len(csv_content) - 2000} more characters)")

            except PermissionError:
                print("\n⚠️  PERMISSION ERROR: Cannot write files in current directory.")
                print("   Files may be open in another program or you lack write permissions.")
                print("\n📋 CSV CONTENT (copy-paste this to save manually):")
                print("="*80)
                csv_content = assignable[['Lead No.', 'Priority Tier', 'Follow-up Timing', 'Reason']].to_csv(index=False)
                print(csv_content)

            print("\n" + "="*80)
            print("🎯 TELE-SALES TEAM RECOMMENDATIONS:")
            print("="*80)
            print("🥇 PRIORITY TIER 1: Leads with 'proposal sent' + 'will let me know' / Payment secured")
            print("   → Highest conversion potential. Follow up within 24 hours.")
            print("   → Focus on payment confirmation and closing the deal.")
            print()
            print("🥈 PRIORITY TIER 2: Leads with active hiring + contact details shared")
            print("   → Strong engagement. Follow up within 48 hours.")
            print("   → Share proposals and schedule meetings.")
            print()
            print("🥉 PRIORITY TIER 3: Leads marked 'busy' but committed to callback")
            print("   → Respect stated times (e.g., 'call after 3pm', 'tomorrow').")
            print("   → Set calendar reminders for exact callback times.")
            print()
            print("📝 LOW PRIORITY: Other valid leads")
            print("   → Follow up weekly or as capacity allows.")
            print("="*80)

        else:
            print("\n⚠️  NO ASSIGNABLE LEADS FOUND. Check:")
            print("   - Memo content quality")
            print("   - Keyword patterns in classify_lead() function")
            print("   - Column name mappings")

        # Pattern analysis (top reasons for disqualification)
        if len(df) - len(assignable) > 0:
            print("\n🔍 COMMON DISQUALIFICATION PATTERNS:")
            reason_counts = df[df['Assignment Status'] == 'Not Assignable']['Reason'].value_counts().head(5)
            for reason, count in reason_counts.items():
                print(f"   • {reason}: {count:,} leads")

        print("\n" + "="*80)

    except FileNotFoundError:
        print("❌ ERROR: 'All Lead Memo Only Data.csv' not found in current directory.")
        print("   Place your file here and rerun the script.")
        sys.exit(1)
    except Exception as e:
        print(f"❌ UNEXPECTED ERROR: {type(e).__name__}: {str(e)}")
        import traceback
        traceback.print_exc()
        sys.exit(1)

if __name__ == "__main__":
    main()

C:\Users\Asus\AppData\Local\Temp\ipykernel_27692\2128429557.py:165: DtypeWarning: Columns (0: Company ID, 1: 1st time(V), 2: 2nd time(V), 3: 3rd time(V), 4: 5th time(V), 5: Contact No.3, 6: Contact No.4, 7: Flag?, 8: Have You Login, 9: Area(old), 10: Hiring Status for Bdjobs, 11: Meeting Done Date, 12: Do you use a CV bank?, 13: Are you looking for a job?) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('C:/Users/Asus/All Lead Memo Only Data.csv', encoding='latin1')


🚀 TELE-SALES LEAD PRIORITIZATION SYSTEM

📊 Total leads analyzed: 31,135
✅ Assignable to tele-sales: 2,091 (6.7%)
❌ Not assignable: 29,044

🎯 PRIORITY TIER DISTRIBUTION:
--------------------------------------------------------------------------------
🥇 Tier 1          :    127 leads (  6.1%)
🥈 Tier 2          :    204 leads (  9.8%)
🥉 Tier 3          :    378 leads ( 18.1%)
📝 Low Priority    :  1,382 leads ( 66.1%)

📋 DETAILED LEAD BREAKDOWN:

Tier 1 LEADS (127 total):
--------------------------------------------------------------------------------
Lead No.: 353      | Priority: Tier 1     | Timing:                     
          Memo: Payment done for 1 slot
--------------------------------------------------------------------------------
Lead No.: 353      | Priority: Tier 1     | Timing:                     
          Memo: Random visit done for selling our free package. But fortunately this was a hot l...
-------------------------------------------------------------------------------